# 2.3 · Linear algebra: how a model combines numbers

**Estimated time:** 40–50 minutes · **Prerequisite:** Lesson 2.1 (tensors and shape).

### By the end, you should be able to

- Distinguish a **scalar**, **vector**, **matrix**, and their shapes.
- Calculate a **dot product** by hand and interpret it as a weighted sum.
- Explain a **matrix–vector product**: one prediction per example.
- Explain the shape rule for **matrix multiplication**, and distinguish it from elementwise multiplication.
- Understand **sum**, **mean**, and the **L2 norm** without memorizing formal proofs.

**Running example:** We will use Hours from the fictional learner table to calculate made-up quiz predictions. We deliberately choose a simple rule so each result can be checked by hand.


## 1. Why linear algebra matters to machine learning

Imagine predicting a quiz score from study hours. A simple model could follow the rule:

\[\widehat{\text{Score}} = 2\times \text{Hours} + 1.\]

The **2** is the weight for Hours; the **1** is a bias (constant offset). A larger neural network uses the same essential idea—combine many inputs with learned weights—repeated across many layers.

A convenient way to represent this small example is with two **features** per learner: the number of hours and a constant `1` for the bias. The weights are `[2, 1]`.


## 2. The dot product: one weighted sum

For two vectors of equal length, the **dot product** multiplies matching entries and **adds the products**. Given `x = [2, 4]` and `w = [3, 1]`:

\[x\cdot w = 2\times3 + 4\times1 = 10.\]

Think: each feature contributes its value multiplied by its weight. The final sum is one number (a scalar).


### Why multiplication *and* addition?

A model may consider several pieces of information, each with a different importance or **weight**. The dot product gives one combined value: multiply each feature by its corresponding weight, then add those contributions.

For `[2,4] · [3,1]`: feature 1 contributes `2×3=6`, feature 2 contributes `4×1=4`, and the sum is **10**. The answer is a single scalar, not `[6,4]`.

**ML connection:** a single neuron begins with a weighted sum like this (often plus a bias), before applying its activation function. The weights are learned, not measures of a feature's inherent worth.


<picture>
  <source type="image/svg+xml" srcset="images/09-dot-product.svg">
  <img src="images/09-dot-product.png" alt="A dot product with two explicitly labeled vectors and the full 2×3 + 4×1 calculation" loading="lazy">
</picture>

*Figure: A dot product with two explicitly labeled vectors and the full 2×3 + 4×1 calculation.*


In [ ]:
import torch

x = torch.tensor([2., 4.])
w = torch.tensor([3., 1.])

manual = x[0]*w[0] + x[1]*w[1]
print('Hand-style calculation:', manual.item())
print('PyTorch dot:', torch.dot(x,w).item())
assert torch.dot(x,w).item() == 10.


**Do not confuse `dot` with elementwise multiplication.** `x*w` returns `[6, 4]`, still a vector. `torch.dot(x,w)` **adds** those two values to return the scalar `10`.

A dot product is a building block of predictions, attention scores in Transformers, and many other neural-network operations.


In [ ]:
print('Elementwise x*w:', (x*w).tolist())
print('Dot product x·w:', torch.dot(x,w).item())


## 3. Matrix–vector multiplication: several predictions at once

Instead of predicting one learner at a time, stack the feature vectors as rows of a matrix. Our four learners have Hours `[1,2,3,4]`. For each learner, we append a constant `1`, because the rule includes a bias.

\[
X = \begin{bmatrix}1&1\\2&1\\3&1\\4&1\end{bmatrix},\quad
w=\begin{bmatrix}2\\1\end{bmatrix},\quad
Xw=\begin{bmatrix}3\\5\\7\\9\end{bmatrix}.
\]

Every **output value** is a **row of X dot w**. The first learner gets `1×2 + 1×1 = 3`, the second `2×2 + 1×1 = 5`, and so on.

**Shape rule:** `(4, 2) @ (2, 1) → (4, 1)`. The **inner dimensions** (both 2) must match, and the **outer dimensions** (4 and 1) give the output shape.


### Read matrix multiplication one row at a time

The first row of `X` is `[1,1]`. Dot it with weights `[2,1]` to obtain `1×2 + 1×1 = 3`. The second row `[2,1]` gives `2×2+1×1=5`, then rows 3 and 4 produce 7 and 9.

**Why is this useful?** One matrix operation makes *four predictions* without writing four separate formulas. A matrix shaped `(number_of_examples, number_of_features)` multiplied by a weight column shaped `(number_of_features, 1)` gives one prediction per example. If the numbers of features do not agree, the multiplication is undefined.


<picture>
  <source type="image/svg+xml" srcset="images/10-matrix-vector.svg">
  <img src="images/10-matrix-vector.png" alt="A 4 × 2 feature matrix times a 2 × 1 weight vector producing all four predictions in a 4 × 1 output grid" loading="lazy">
</picture>

*Figure: A 4 × 2 feature matrix times a 2 × 1 weight vector producing all four predictions in a 4 × 1 output grid.*


In [ ]:
X = torch.tensor([[1.,1.], [2.,1.], [3.,1.], [4.,1.]])
weights = torch.tensor([[2.], [1.]])
predictions = X @ weights
print('X shape:',tuple(X.shape))
print('Weights shape:',tuple(weights.shape))
print('Predictions shape:',tuple(predictions.shape))
print('Predicted scores:',predictions.squeeze(1).tolist())
assert predictions.squeeze(1).tolist() == [3.,5.,7.,9.]


## 4. Matrix–matrix multiplication: what changes?

The same rule works when the second operand is a *matrix* rather than one weight vector. If `A` has shape `(2, 3)` and `B` has shape `(3, 2)`, their product has shape `(2, 2)`.

To find **one output cell**, take a row from `A` and a column from `B` and calculate their dot product. The top-left output, for example, is `1×1 + 2×0 + 3×1 = 4`. The diagram below includes **every** input and output cell. Check another output cell yourself.


### Check every output cell by hand

For `A = [[1,2,3],[4,5,6]]` and `B = [[1,0],[0,1],[1,1]]`, the result is:

| | B column 0 | B column 1 |
|:--|--:|--:|
| A row 0 | `1×1 + 2×0 + 3×1 = 4` | `1×0 + 2×1 + 3×1 = 5` |
| A row 1 | `4×1 + 5×0 + 6×1 = 10` | `4×0 + 5×1 + 6×1 = 11` |

Thus `A @ B = [[4,5],[10,11]]`, with shape `(2,2)`. **Matrix multiplication is not matching-cell multiplication:** it combines a row with a column and sums across the shared inner dimension.


<picture>
  <source type="image/svg+xml" srcset="images/11-matrix-matrix.svg">
  <img src="images/11-matrix-matrix.png" alt="Fully labeled 2 × 3 and 3 × 2 matrices with a complete 2 × 2 result and an annotated cell calculation" loading="lazy">
</picture>

*Figure: Fully labeled 2 × 3 and 3 × 2 matrices with a complete 2 × 2 result and an annotated cell calculation.*


In [ ]:
A = torch.tensor([[1.,2.,3.],[4.,5.,6.]])
B = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
C_result = A @ B
print(C_result)
print('Result shape:',tuple(C_result.shape))
assert C_result.tolist() == [[4.,5.],[10.,11.]]


### Three symbols to remember

| Python expression | Meaning | Example result |
|:--|:--|:--|
| `a + b` | add matching entries (or broadcast) | `[1,2]+[3,4] → [4,6]` |
| `a * b` | multiply matching entries (or broadcast) | `[1,2]*[3,4] → [3,8]` |
| `A @ B` | row–column dot products | `(2,3)@(3,2) → (2,2)` |

This distinction becomes critical when reading neural-network formulas like `X @ W + b`.

## 5. Reductions: turning many values into fewer values

A **reduction** summarizes a collection. `sum()` adds; `mean()` averages. Along `axis=0`, we summarize **down the rows**, getting one value per column. Along `axis=1`, we summarize **across columns**, getting one value per row.

For the matrix `[[1,2],[3,4]]`, column sums are `[4,6]`; row sums are `[3,7]`.


### Which axis disappears?

Take `[[1,2],[3,4]]`, shape `(2,2)`. `sum(dim=0)` adds **down** each column and returns `[4,6]`. `sum(dim=1)` adds **across** each row and returns `[3,7]`. A *reduction* decreases the number of axes or their sizes by summarizing values.

**ML connection:** averaging a batch of losses turns one loss per example into one batch-level training objective. Choosing the wrong axis can silently produce the wrong result even if the program runs.


In [ ]:
small = torch.tensor([[1., 2.], [3., 4.]])
print('All entries sum:', small.sum().item())
print('Column sums (dim=0):', small.sum(dim=0).tolist())
print('Row sums (dim=1):', small.sum(dim=1).tolist())
print('Mean of all entries:',small.mean().item())


## 6. Norms: measuring vector size

The **L2 norm** measures a vector's length. For `v = [3, 4]`, picture a right triangle with horizontal side 3 and vertical side 4. Pythagoras tells us the diagonal has length 5.

\[\|v\|_2=\sqrt{3^2+4^2}=5.\]

This matters because we often want to measure the size of a parameter vector, error vector, or gradient. You do not need advanced geometry to recognize what a norm tells you.


### Norm versus distance

For `v = [3,4]`, the **L2 norm** is `√(3²+4²)=5`: distance from the origin. To measure the distance *between* two vectors `a` and `b`, first take the difference `a−b`, then its norm `||a−b||₂`.

**Why we care:** gradient clipping and weight regularization can use norms to control the magnitude of parameter vectors or updates; a norm measures size, not whether a prediction is correct.


<picture>
  <source type="image/svg+xml" srcset="images/12-vector-norm.svg">
  <img src="images/12-vector-norm.png" alt="A 3–4–5 right triangle illustrating the length of a vector with entries 3 and 4" loading="lazy">
</picture>

*Figure: A 3–4–5 right triangle illustrating the length of a vector with entries 3 and 4.*


In [ ]:
v = torch.tensor([3.,4.])
print('L2 norm:', torch.linalg.vector_norm(v).item())
print('Squared entries:',(v*v).tolist())
assert torch.linalg.vector_norm(v).item() == 5.


## 7. Small experiment: change one weight

Our simple predictor uses `[weight_for_hours, bias] = [2,1]`. What if the Hours weight becomes **3**? Higher hours would have more effect on the prediction, because they are multiplied by a larger number.

Try predicting the four scores **before** running the cell. (They will no longer match the original synthetic targets; that is useful to notice.)


In [ ]:
new_weights = torch.tensor([[3.],[1.]])
changed_predictions = X @ new_weights
print('Previous:',predictions.squeeze().tolist())
print('New:',changed_predictions.squeeze().tolist())


### Check your understanding

1. Calculate `[3, 2] · [2, 5]` by hand. Is the result a number or a vector?
2. What output shape should you expect from `(5, 3) @ (3, 2)`? Why?
3. When is `A * B` **not** the same as `A @ B`?
4. What is the L2 norm of `[0, 6, 8]`?
5. In the prediction `2×Hours + 1`, which part is the **bias**?

**Takeaway:** Linear algebra lets models combine many numerical features using weighted sums, while tensor shape tells us exactly which calculations are valid.

➡️ **Next:** [2.4 Calculus and automatic differentiation](04-calculus-and-autograd.ipynb) — how does the model know which weight to change?
